# Publisher Application

This application produces Confluence content based on an information model. 

1. Setup and Configuration 
1. Build page tree based on configuration and from information model (SSOT JSON)
1. Create content (page, attachments, ...) including translations
1. Back up existing content (Optional)
1. Upload/publish content

In [ ]:
import os
import sys

## Configuration
Be aware not to commit your credentials!

In [ ]:
configuration_file = 'geberit.yaml'
assert os.path.isfile(configuration_file), 'Cannot read file ' + configuration_file

### Load configuration

In [ ]:
import yaml
import copy

with open(configuration_file) as f:
    config = yaml.safe_load(f)

conf_conf = config['confluence']
assert conf_conf
assert len(conf_conf['apiurl']) > 0
space_key = conf_conf['space']
root_page = conf_conf['rootpage']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# this is the parameter cell. see cell tags. used to overwrite things with test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']
cooldown = config['confluence'].get('cooldown', 0.0)
disclaimer = config.get('disclaimer', '')

## Initialize logging

In [ ]:
import logging

log = logging.getLogger()
log.setLevel(logging.INFO)

LOGFILE = 'target/debug.log'
os.makedirs('target', exist_ok=True)

handler = logging.handlers.RotatingFileHandler(
    LOGFILE, maxBytes=(1048576*5), backupCount=7
)
formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)
handler.setLevel(logging.DEBUG)
log.addHandler(handler)

log.info('Starting execution')

## Bootstrap Confluence integration

The [Confluence API](https://github.com/atlassian-api/atlassian-python-api) is embedded as a **git submodule** in the 'lib' folder next to this notebook.

Use `git submodule update --init` to fetch all submodules after a checkout without `--recursive` option.

If the next cell fails, install confluence-api submodule from the repository root with:
`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/contentfactory/lib/atlassian-python-api`

In [ ]:
library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence

In [ ]:
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password, cloud=True)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])
assert root_page_id, "Cannot find documentation root page '{}' in space {} on server {}".format(config['confluence']['rootpage'], space_key, config['confluence']['apiurl'])
root_page_id

# Setup translation

Requires gettext: `conda install gettext`

Trigger scan for translatable objects:
    `xgettext --from-code utf-8 -L python -d scan.pot templates/*`
    `msgmerge --width --update

https://phrase.com/blog/posts/translate-python-gnu-gettext/

## Internationalisation (i18n)

Publishing of content in different languages is supported. The GNU `gettext` toolchain is used to translate texts in code and templates.

Translation files are located in the 'locale' directory structure.

Use `xgettext --from-code utf-8 -L python -d confluence-publisher templates/*` to scan for content and
`msgfmt -o ../locale/de/LC_MESSAGES/confluence-publisher.mo ../locale/de/LC_MESSAGES/confluence-publisher.po` to upate the compiled translation files.

Implemtation in section 'Translation'

In [ ]:
import gettext
locale_folder = config.get('locale', './locale')
gettext.bindtextdomain('confluence-publisher', locale_folder)

In [ ]:
from pathlib import Path
translations_folders = list(Path(locale_folder).rglob("LC_MESSAGES"))
translations_folders

## Translator

Translation is done using the custom translator
It uses gettext translate when there is no translation provided in the SSOT.



In [ ]:
class Translator:
    """Translate strings"""
    logger = logging.getLogger(__name__)
    
    def __init__(self, language: str):
        self.language = language
        self.translator = gettext.translation('confluence-publisher', config.get('locale', './locale'), fallback=True, languages=[language])
        self.title_format = '{title} - [{language}]'
        
    def tr(self, element) -> str:

        if isinstance(element, dict):
            """If the provied value is a field containing translations, use them"""
            text = element.get(self.language)
            if not text and len(element.values()) > 0:
                text = list(element.values())[0]
                self.logger.warning('Falling back to {} from {}'.format(text, str(element)))
            if not text:
                return ''
            return text

        if isinstance(element, str):
            translated = self.translator.gettext(element)
            return translated

        self.logger.warning('Cannot translate element "{}" of type {}'.format(element, type(element)))
        return ''
    
    def title_language(self, title: str) -> str:
        """Create unique confluence page title per translation"""
        return self.title_format.format(title = title, language = self.language)
    
    def key_lang(self, key: str) -> str:
        return key + '-' + self.language
    
    def lang(self) -> str:
        return self.language

In [ ]:
translators = { language: Translator(language) for language in config['languages'] }
translators

In [ ]:
default_language_translator = translators[config['languages'][0]]
default_language_translator.title_format = '{title}'
'Default language is {}'.format(default_language_translator.lang())

### Selftests

In [ ]:
default_language_translator.tr('fadsjfklj4q9u')

In [ ]:
default_language_translator.tr('Synonyms')

In [ ]:
default_language_translator.tr({ 'es': 'hola'} )

In [ ]:
default_language_translator.tr('Attributes')

## Load the data
The **data** is the JSON serialized information model 

In [ ]:
import json

assert os.path.isfile(config['json']), 'The datasource in config.json {} is not a file'.format(config['json'])

data = None
with open(config['json'], 'r') as source:
     data = json.load(source)

str(data)[:512]

In [ ]:
model_languages = list(data['languages'])
config_languages = list(config['languages'])
'Model languages: {}. Export configuration selected languages: {}'.format(model_languages, config_languages)

In [ ]:
# Missing languages?
missing_translations = set(config_languages) - set(model_languages)
assert len(missing_translations) == 0, 'Missing languages in model: {}'.format(missing_translations) 

In [ ]:
categories = list(data)
for category in categories:
    print('{} Elements in category "{}"'.format(len(data[category]), category))

## Navigator
The navigator contains the map of all pages. It allows create confluence links from one page to it's translations and to parent pages.
The internal page map contains:
{ 'key': element key without language suffix  'page': page dictionary, 'confluence_id': if published } 

In [ ]:
class Navigator:
    
    def __init__(self):
        self.page_map = {}
        self.page_uniqueness_map = {}
    
    def register_page(self, key: str, lang: str, page: dict):
        self.page_map[self.title_lang(key, lang)] = page
        self.page_uniqueness_map[page['title']] = page
    
    def title_lang(self, key: str, lang: str) -> str:
        return key + '-' + lang
    
    def page_4_language(self, key: str, lang: str) -> str:
        return self.page_map.get(self.title_lang(key, lang))
    
    def translation_title(self, key: str, lang: str) -> str:
        """Get the title of the page with 'key' for language 'lang'"""
        return self.translation_page(key, lang)['title']

    def pages(self) -> list:
        return list(self.page_map.values())
    
    def check_page_title(self, title: str) -> bool:
        return self.page_uniqueness_map.get(title)

## Prepare destination structure
Configuration:
- content
  - Systems
    - Tables
      - Columns

Rolled out:
- Topic 'Systems'
  - System A
    - Table A1
      - Column ID - A1
      - Colunn Name - A1
      - Column Value - A1
    - Table A2
  - System B
    - Table B1
    - Table B2

In [ ]:
def ensure_unique_page_title(page: dict, navigator: Navigator):
    """Make sure the page title is unique in this Confluence space. This does eventually modify the page element!"""
    title = page['title']
    existing_page_with_same_title = navigator.check_page_title(title)
    if existing_page_with_same_title:
        print('Page for {}[{}] has same title as {}[{}]\nOld:{}\nNew:{}'.format(
            existing_page_with_same_title['key'], existing_page_with_same_title['title'], page['key'], title,
            existing_page_with_same_title['item'], page['item']))
        tokens = title.split('-')
        if len(tokens) > 1:
            tokens.insert(len(tokens) - 1, ' ' + page['key'] + ' ')
        else:
            tokens.append(' ' + page['key'] + ' ')
        page['title'] = '-'.join(tokens)
        print('Created unique title {}'.format(page['title']))


def manual_documentation(parent_page: dict, config: dict, navigator: Navigator, translator: Translator):
    """Add a page for manual documentation, if configured"""
    md = config.get('manual-documentation')
    if md:
        page = copy.copy(parent_page)
        page_key = parent_page['key'] + '-manual-documentation'
        page['key'] = page_key
        title_format = md.get('title_format', '{parent_title} - manual')
        page['name'] = title_format.format(parent_name=parent_page['name'], key=parent_page['key'], lang=translator.lang(), parent_title=parent_page['title'])
        page['title'] = translator.title_language(page['name'])
        page['config'] = md
        page['template'] = md.get('template', 'Manual documentation for {title} [{key}]'.format(title=parent_page['title'], key=page['key']))
        # register child with parent
        parent_page['manual-documentation-page-title'] = page['title']
        
        labels = set(parent_page['labels'])
        labels.update(md.get('labels', ['manual-documentation']))
        page['labels'] = labels
        # keep confluence content
        page['preserve'] = True
        page['level'] = page['level'] + 1
        page['template'] = md.get('template')
        
        ensure_unique_page_title(page, navigator)
        navigator.register_page(page_key, translator.lang(), page)
        return page
    else:
        return None


def prepare_pages(parent_page: dict, config: dict, topic: str, level: int, navigator: Navigator, translator: Translator, data: object):
    elements = data[topic]
    pages = []
    item_filter = config.get('filter')
    filtered = 0
    for element_key in list(elements):
    
        item = data[topic][element_key]
        if item_filter:
            filter_result = eval(item_filter)
            if not filter_result:
                filtered += 1
                continue

        name_translated = translator.tr(item['name']).strip()
        if len(name_translated) > 1:
            log.warning('Strange name for item {}'.format(item))

        title_safe = name_translated
        
        if topic == 'attributes':
            parent_key = item['entity']
        elif topic == 'tables':
            parent_key = item['interface-id']
        elif topic == 'columns':
            parent_key = item['table-id']
        else:
            parent_key = parent_page['key']
        
        parent = navigator.page_4_language(parent_key, translator.lang())
        parent['child-count'] = parent['child-count'] + 1
        parent_name = parent['name']
        
        item_title_rule = config.get('title_rule')
        if item_title_rule:
            title_by_rule = eval(item_title_rule)
            if title_by_rule:
                parent_title = title_by_rule
        else:
            # Default naming rule for nested elements: {child_title} - 
            if topic in ['attributes', 'tables', 'columns']:
                title_safe = '{child_title} - {parent_name}'.format(child_title=name_translated, parent_name=parent_name)
                                              
        title = title_safe
        
        labels = set(config.get('labels', []))
        page = {
            'key': element_key,
            'topic': topic,
            'name': title,
            'title': translator.title_language(title),
            'parent': parent,
            'labels': list(labels),
            'item': item,
            'config': config,
            'level': level,
            'translator': translator,
            'template': config.get('template'),
            'child-count': 0,
        }
        
        ensure_unique_page_title(page, navigator)
        
        navigator.register_page(element_key, translator.lang(), page)
        pages.append(page)
        
        md = manual_documentation(page, config, navigator, translator)
        if md:
            pages.append(md)

    print('Added {} pages for topic {}. Filtered out {}'.format(len(pages), topic, filtered))
    
    # Descend into children, if any ...
    child_configurations = config.get('content')
    if child_configurations:
        """Process child types"""
        for child_config_topic in child_configurations:
            if not data[child_config_topic]:
                log.warning('No data for topic "{}"'.format(child_config_topic))
                continue
            child_config = child_configurations[child_config_topic]
            subpages = prepare_pages(page, child_config, child_config_topic, level + 1, navigator, translator, data)

    return pages


def top_level_content(config: dict, navigator: Navigator, translator: Translator, data: dict) -> list:
    """Recurse configuration content structure"""
    content = config.get('content')
    pages = []
    if content:
        for topic_key in list(content):
            topic_config = content[topic_key]
            labels = set(topic_config.get('labels', []))
            labels.add('im-parent')
            labels.add('im-parent-' + topic_key)            

            name = translator.tr(topic_config.get('title')) + ' - ' + data['model']['name']
            category_page = {
                'topic': topic_key + '-root',
                'key': topic_key,
                'name': name,
                'title': translator.title_language(name),
                'labels': list(labels),
                'config': topic_config,
                'parent': None,
                'level': 0,
                'child-count': 0,
                'translator': translator,
                'template': topic_config.get('index-template', 'parent-page-index.templ.html'),
            }
            
            ensure_unique_page_title(category_page, navigator)

            navigator.register_page(topic_key, translator.lang(), category_page)
            pages.append(category_page)
            
            print('Processing category {} [{}]'.format(topic_config.get('title'), topic_key))
                        
            sub_pages = prepare_pages(category_page, topic_config, topic_key, 1, navigator, translator, data)
            if len(sub_pages) == 0:
                category_page['skip'] = True
    else:
        log.error('Cannot find content on root level')
    return pages

In [ ]:
navigator = Navigator()

for lang in config['languages']:
    print('*** Scanning for language {} ***'.format(lang))
    pages = top_level_content(config, navigator, translators[lang], data)
    
total = len(navigator.pages())
'Will produce {} * {} ~= {} pages'.format(len(config['languages']), len(pages), total)

In [ ]:
level0 = list(filter(lambda page: page['level'] == 0, navigator.pages()))
[ page.get('title') for page in level0 ]

In [ ]:
skiplist = list(filter(lambda page: page.get('skip', False), navigator.pages()))
'Will skip {} pages: {} ...'.format(len(skiplist), [page.get('title') for page in skiplist[:5]])

# Create graphs and render the content

In [ ]:
from tqdm.autonotebook import tqdm
from tqdm.notebook import tqdm_notebook

In [ ]:
content_root = Path('confluence-content')
destination_folder = os.path.join(content_root, 'pages')
print('Writing confluence content to disk: {}'.format(destination_folder))
os.makedirs(destination_folder, exist_ok=True)

## Helper class to simplify template rendering

This helper is available in jina2 templates with the name 'util'

In [ ]:
import html
import markupsafe
from functools import reduce

class ConfluenceContentUtil:
    """This util is used in jinja2 scripts to create Confluence content.
    It is designed to provide complex functionality, that does not fit into templates directly."""
    def __init__(self, navigator: Navigator, translator: Translator, languages: list, json_data: dict):
        self.translator = translator
        self.navigator = navigator
        self.language = translator.language
        assert len(self.language) == 2
        self.other_lang = list(languages)
        self.other_lang.remove(self.language)
        assert len(self.other_lang) + 1 == len(languages)
        self.json_data = json_data
    
    def translate(self, text):
        return self.translator.tr(text)
    
    def translate_text(self, field) -> markupsafe.Markup:
        text = html.escape(self.translator.tr(field))
        linebreaks = text.replace('\n', '<br/>\n')
        return markupsafe.Markup(linebreaks)
        
    def other_languages(self) -> list:
        return self.other_lang
    
    def soft_link(self, key: str, lang: str = None) -> markupsafe.Markup:
        """Returns a confluence link if the key is represented with a page in the same language context or the language provided"""
        if key:
            language = lang if lang else self.language
            page = self.navigator.page_4_language(key, language)
            if page:
                return markupsafe.Markup('''<ac:link><ri:page ri:content-title="{page_title}"/><ac:plain-text-link-body><![CDATA[{name}]]></ac:plain-text-link-body></ac:link>'''.format(
                    page_title=html.escape(page['title']), name=html.escape(page.get('name'))))
            else:
                if 'DOMA' in key:
                    return translator.tr(self.json_data['domains'][key]['name'])
                elif 'COLUMN' in key:
                    return translator.tr(self.json_data['columns'][key]['name'])
                else:
                    return key
        else:
            return ''
        
    def relation_self(self, entity_key: str, relation_key: str):
        """Returns the local end of the relation_key attached to enitity_key"""
        relation = self.json_data['relations'][relation_key]
        if relation['from-to']['enti'] == entity_key:
            return relation['from-to']
        else:
            return relation['to-from']

    def relation_other(self, entity_key: str, relation_key: str):
        """Returns the remote end of the relation_key"""
        relation = self.json_data['relations'][relation_key]
        if relation['from-to']['enti'] == entity_key:
            return relation['to-from']
        else:
            return relation['from-to']
        
    def column_lineage(self, column_key: str):
        """Collects columns that are mapped with the column provided via the IM"""
        column = self.json_data['columns'][column_key]
        result = []
        for attribute_key in column['attributesmapped']:
            attribute = self.json_data['attributes'][attribute_key]
            columns_mapped = attribute['columnsmapped+']
            all_columns = map(lambda entry: columns_mapped[entry], columns_mapped)
            cols = reduce(lambda e, l: e + l, list(all_columns), [])
            result.extend(cols)
        try:
            result.remove(column_key)
        except ValueError:
            # fine if it is not in the list
            pass
        return result
    
    def attribute_lineage(self, attribute_key: str):
        """Collects columns that are mapped to the provided attribute"""
        attribute = self.json_data['attributes'][attribute_key]
        columns_mapped = attribute['columnsmapped+']
        all_columns = map(lambda entry: columns_mapped[entry], columns_mapped)
        cols = reduce(lambda e, l: e + l, list(all_columns), [])
        return cols
    
    def icon(self, key: str) -> markupsafe.Markup:
        return markupsafe.Markup(
            '<img width="50px" align="right" ' +
            'src="http://res.cloudinary.com/foryouandyourcustomers/image/upload/fyayc_icon_library/svg/0099.svg" />'
        )

In [ ]:
test = ConfluenceContentUtil(navigator, translators['de'], ['en','fr','de'], data)
test.other_languages()

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

def create_jinja2_i18n_env(lang:str) -> Environment:
    jinja_env = Environment(
        loader=FileSystemLoader('./templates'),
        autoescape=select_autoescape(['html', 'xml']),
        extensions=["jinja2.ext.i18n"])
    
    translator = translators[lang]
    jinja_env.install_gettext_translations(translator.translator, newstyle=True)    
    util = ConfluenceContentUtil(navigator, translator, config['languages'], data)
    
    jinja_env.globals.update({ 'util': util, 'disclaimer': disclaimer, 'jinja_env': jinja_env, 'i18n': util })
    return jinja_env

i18n_environments = { lang: create_jinja2_i18n_env(lang) for lang in config['languages'] }

## Graph rendering

In [ ]:
import sys
import os
import glob
sys.path.insert(0, os.path.abspath('../../pythonWork/pythonSource'))
sys.path.insert(1, os.path.abspath('../../pythonWork/pythonSource/IM_db'))
import IM_db.createDB as cdb
import IM_ODM.fillDB as fdb
from IM_JSON import JSModel
from IM_WEB.IM_HTML import entityenviron

In [ ]:
jsmodel = JSModel.readfromfile(pfilename=config['json'])

In [ ]:
def create_graphs(page: dict, destination_folder) -> str:
    if page.get('topic') == 'entities':
        translator = pmodellang=page['translator']
        svg_env = entityenviron.createentienvironment(pentiid=page['key'], pjson=jsmodel, pmodellang=translator.lang())
        svg_node = entityenviron.generate_svg_content(penviron=svg_env)
        #svg_node = svg_node.replace('version="1.1" ', 'version="1.1" viewBox="0 0 450 450"')
        full_svg = '<?xml version="1.0" encoding="UTF-8"?>' + '\n' + svg_node
        svg_file_name = translator.key_lang(page['key']) + '.svg'
        svg_target = os.path.join(destination_folder, svg_file_name)
        with open(svg_target, 'w') as out:
            out.write(full_svg)
        page['entity-graph-name'] = svg_file_name
        page['entity-graph-path'] = svg_target
        return svg_target

In [ ]:
entity0_key = list(data['entities'])[0]
page = { 'topic': 'entities', 'key': entity0_key, 'translator': translators['de']}
test = create_graphs(page, destination_folder)
print('Wrote ' + test)

# Generate content

In [ ]:
import re

def write_page_to_disk(page, content: str, destination_folder):
    target_file = os.path.join(destination_folder, page['key'] + '-' + page['translator'].lang() + '.chtml')
    with open(target_file, 'w') as out:
        out.write(content)
    page['content'] = content
    return target_file

pages = list(navigator.pages())
with tqdm_notebook(total=len(pages), dynamic_ncols=True, unit='Page') as pbar:
    for page in pages:
        element_config = page['config']
        
        # todo generate graph here
        create_graphs(page, destination_folder)
        
        template_file_name = page.get('template')
        if template_file_name:
            translator = page['translator']
            jinja_env = i18n_environments[translator.language]
            jinja_template = jinja_env.get_template(template_file_name)
            
            rendered = jinja_template.render(page=page, data=data, key=page.get('key'), item=page.get('item'), 
                config=element_config, update_message = 'update')
            
            content = re.sub('<!--.*?->(\n)*', '', rendered) # strip comment lines
            ondisk = write_page_to_disk(page, content, destination_folder)
            page['file'] = ondisk

            
        pbar.update(1)

In [ ]:
from requests.exceptions import HTTPError
from colorama import Fore, Style

def print_http_error_details(message: str = None, page: dict = None, exception: HTTPError = None):
    print(Fore.RED + exception.response.content.decode('utf-8'))
    from pprint import pprint
    print(Fore.YELLOW + str(vars(e)))
    pprint(vars(exception.response))
    result = exception.response.raw
    pprint(vars(result))
    if page:
        print('Page {title} {id}'.format(id=page.get('key'), title=page.get('title')))
    print(Style.RESET_ALL)
    log.error('{message}\nContext: {title} [{key}]'.format(message=message, title=page.get('title'), key=page.get('key')), exception)


## Cautious mode
Scan the space before publishing to prevent overwrite or move of pages

In [ ]:
from atlassian.confluence import ApiError

archive_folder = os.path.join(content_root, 'archive')
os.makedirs(archive_folder, exist_ok=True)

def preserve_page(page_id: str, page: dict):
    page['confluence-page-id'] = page_id
    try:
        content = confluence.get_page_by_id(page_id, expand='body.storage,ancestors,version,history')
        page['previous'] = content
        ancestors = content.get('ancestors')
        page['ancestors'] = ancestors
        current_content = content['body']['storage']['value']
        page['previous-content'] = current_content
        ancestor_ids = set(map(lambda a: a['id'], ancestors))
        ancestor_titles = list(map(lambda a: a['title'], ancestors))
        if not root_page_id in ancestor_ids:
            log.warning("Will move page with title {title} ID={page_id} from hierarchy:\n{ancestors}".format(title=page['title'], page_id=page_id, ancestors='/'.join(ancestor_titles)))
        with open(os.path.join(archive_folder, page['translator'].key_lang(page['key']) + '.chtml'), 'w') as out:
            out.write(current_content)
    except HTTPError as e:
        print_http_error_details("Cannot preserve current content", page, e)
        
def scan_page(page: dict):
    title = page['title']
    try:
        page_id = confluence.get_page_id(space_key, title)
        if not page.get('preserve', False):
            log.warning("Page with title {title} exists. ID={id}".format(title=title, id=page_id)) 
        preserve_page(page_id, page)
        return page
    except ApiError as e:
        # It is perfectly safe, that a page does not exist
        pass

In [ ]:
p = list(navigator.pages())[330]
x = scan_page(p)
assert x

In [ ]:
x.get('content')

In [ ]:
x.get('previous-content')

In [ ]:
pages = list(navigator.pages())
print('Scanning {} pages in confluence space {}'.format(len(pages), space_key))
existing_pages = []
with tqdm_notebook(total=len(pages), dynamic_ncols=True, unit='Page', smoothing=0.1) as pbar:
    for page in pages:
        page = scan_page(page)
        if page:
            existing_pages.append(page)
        pbar.update(1)

In [ ]:
print('Found {existing} existing pages of {total}'.format(existing=len(existing_pages), totla=len(navigator.pages())))

In [ ]:
existing_pages[0]

In [ ]:
import time
def upload_attachments(page: dict, confluence):
    svg_file_name = page.get('entity-graph-name')
    if svg_file_name:
        sourcefile = page.get('entity-graph-path')
        with open(sourcefile, 'rb') as source:
            content = source.read()
            
        try:
            result = confluence.attach_content(content, name=svg_file_name, content_type='image/svg+xml', 
                                               page_id=page['confluence_id'], space=space_key, comment="Entity-Graph for {}".format(page.get('key')))
            #print('Attachment {}'.format(result))
            page['entity-graph-attachment_result'] = result
            time.sleep(cooldown)
            return result
        except HTTPErrror as e:
            print_http_error_details('Failed to upload attachment ' + svg_file_name, page, e)

In [ ]:
test_page = { 'entity-graph-name': '0593.svg', 'entity-graph-path': 'testdata/0593.svg', 'key': 'test' }
test_page['confluence_id'] = root_page_id
upload_attachments(test_page, confluence)

In [ ]:
def upload(page: dict, content: str):
    parent = page.get('parent')
    if not parent:
        parent_page_id = root_page_id
    else:
        parent_page_id = parent.get('confluence_id')
        assert parent_page_id, 'Missing parent id for page {}'.format(page)
        
    page_id = None
    title = page['title']
    try:
        log.debug('Looking up page {}'.format(title))
        page_id = confluence.get_page_id(space_key, title)
        log.debug('Reading page {}'.format(page_id))
        current_content = confluence.get_page_by_id(page_id, expand='body.storage,ancestors,version,history')
        ancestors = current_content['ancestors']
        current_parent = None
        if len(ancestors) > 0:
            current_parent = ancestors[-1]['id']
        if current_parent != parent_page_id:
            log.warning('Moving page {title} from {source} to {destination}'.format(
                title=title, source=current_parent, destination=parent_page_id))
            confluence.move_page(space_key, page_id, target_id=parent_page_id)
    except Exception as e:
        log.warning('Cannot prepare page "{title}"'.format(title=page.get('title')), e)
        create_result = confluence.create_page(space_key, title=title, parent_id=parent_page_id, body=content)
        page_id = create_result['id']
    
    assert page_id, 'Missing page id for page "{title}"'.format(page.get('title'))
    
    if not page.get('preserve', False):
        try:
            result = confluence.update_page(page_id, title, content, minor_edit=True, version_comment='test')
            page['confluence_result'] = result
        except HTTPError as error:
            print_http_error_details('Failed to update page {title}'.format(title=page.get('title')), page, error)
            page['update_fail_count'] = page.get('update_fail_count', 0) + 1
    
    page['confluence_id'] = page_id
    
    upload_attachments(page, confluence)    
    # TODO upload attachments
    # TODO apply labels
    
    return page['confluence_id']

In [ ]:
level0_pages = list(filter(lambda page: page['level'] == 0, navigator.pages()))
smoketest_page = level0_pages[2]
result = upload(smoketest_page, '+stub+')
cr = smoketest_page['confluence_result']
'Page creation smoke test works. Page created with id {}. See: {}{} {}'.format(result, cr['_links']['base'], cr['_links']['webui'], smoketest_page['confluence_result'])

## Persist page map before upload

In [ ]:
pagelist_copy = copy.deepcopy(navigator.page_map)
for page_key in pagelist_copy:
    page = pagelist_copy[page_key]
    page.pop('translator', None)
    
with open('pagemap.json', 'w') as output:
    json.dump(pagelist_copy, output, indent=2, sort_keys=False)

## Upload parallel

In [ ]:
'Uploading {} pages to {} with cooldown of {}s after each page'.format(len(navigator.pages()), conf_conf['apiurl'], cooldown)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

summary = None

for level in range(0,6):
    level_pages = list(filter(lambda page: page['level'] == level, navigator.pages()))
    print('Level {} has {} pages'.format(level, len(level_pages)))
    with tqdm_notebook(total=len(level_pages), dynamic_ncols=True, unit='Page', smoothing=0.1) as pbar:

        def publish(page):
            if page.get('skip', False):
                log.info('Skipping page {}'.format(page.get('key')))
                pbar.update(1)
                return True
            
            content_file = page.get('file')
            if content_file:
                pbar.set_description('Uploading {title} [{key}]'.format(title=page['title'], key=page['key'], file=content_file))
                try:
                    with open(content_file, 'r') as content:
                        result = upload(page, str(content.read()))
                    time.sleep(cooldown)
                except HTTPError as httpe:
                    print_http_error_details('Cannot publish', page, httpe)
                except Exception as e:
                    print(e)
            else:
                pbar.set_description('Stubbing {}'.format(page['title']))
                result = upload(page, '+stub+')
            pbar.update(1)
            return result

        with ThreadPoolExecutor(max_workers=8) as executor:
            summary = executor.map(publish, level_pages)
        pass

result = list(filter(None, summary))

In [ ]:
sys.modules.keys()

In [ ]:
vars(sys.modules['IM_DB.parameters'])